In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

In [2]:
hitter = pd.read_csv("../data/hitter_training_v4.csv")
pitcher = pd.read_csv("../data/pitcher_training_v4.csv")

print(hitter.shape)
print(pitcher.shape)

(80, 36)
(43, 22)


In [3]:
target = "annual_avg_salary"

drop_cols = [
    "player_name",
    "position",
    "team",

    "annual_avg_salary",

    "total_contract_amount",
    "contract_years"
]

hitter_model_df = hitter.copy()

hitter_features = [
    col
    for col in hitter_model_df.columns
    if col not in drop_cols
]

for col in hitter_features + [target]:
    hitter_model_df[col] = pd.to_numeric(
        hitter_model_df[col],
        errors="coerce"
    )

for col in hitter_model_df.columns:
    if hitter_model_df[col].isnull().sum() > 0:
        hitter_model_df[col] = hitter_model_df[col].fillna(
            hitter_model_df[col].median()
        )

print(hitter_model_df.shape)

(80, 36)


In [5]:
target = "annual_avg_salary"

drop_cols = [
    "player_name",
    "position",
    "team",
    "annual_avg_salary",
    "total_contract_amount",
    "contract_years"
]

hitter_model_df = hitter.copy()

# 숫자로 바꿀 수 있는 컬럼만 모델 후보로 사용
hitter_features = [
    col for col in hitter_model_df.columns
    if col not in drop_cols
]

for col in hitter_features + [target]:
    hitter_model_df[col] = pd.to_numeric(
        hitter_model_df[col],
        errors="coerce"
    )

# 직전 시즌 결측치는 3년 평균으로 대체
if "war_last_year" in hitter_model_df.columns:
    hitter_model_df["war_last_year"] = hitter_model_df["war_last_year"].fillna(
        hitter_model_df["war_3yr_avg"]
    )

if "ops_last_year" in hitter_model_df.columns:
    hitter_model_df["ops_last_year"] = hitter_model_df["ops_last_year"].fillna(
        hitter_model_df["ops_3yr_avg"]
    )

if "wrc_plus_last_year" in hitter_model_df.columns:
    hitter_model_df["wrc_plus_last_year"] = hitter_model_df["wrc_plus_last_year"].fillna(
        hitter_model_df["wrc_plus_3yr_avg"]
    )

# 전부 NaN인 컬럼 제거
hitter_model_df = hitter_model_df.dropna(axis=1, how="all")

# 제거 후 feature 목록 다시 생성
hitter_features = [
    col for col in hitter_model_df.columns
    if col not in drop_cols
]

# 남은 결측치 처리
for col in hitter_features:
    if hitter_model_df[col].isnull().sum() > 0:
        median_value = hitter_model_df[col].median()

        # median도 NaN이면 그 컬럼은 정보가 없으므로 제거
        if pd.isna(median_value):
            hitter_model_df = hitter_model_df.drop(columns=[col])
        else:
            hitter_model_df[col] = hitter_model_df[col].fillna(median_value)

# 최종 feature 목록 다시 생성
hitter_features = [
    col for col in hitter_model_df.columns
    if col not in drop_cols
]

print("타자 모델 데이터:", hitter_model_df.shape)
print("남은 결측치:", hitter_model_df[hitter_features + [target]].isnull().sum().sum())
print("사용 피처 수:", len(hitter_features))
print(hitter_features)

타자 모델 데이터: (80, 34)
남은 결측치: 0
사용 피처 수: 28
['fa_year', 'age_at_fa', 'games_3yr_avg', 'ab_3yr_avg', 'avg_3yr_avg', 'obp_3yr_avg', 'slg_3yr_avg', 'ops_3yr_avg', 'isop_3yr_avg', 'babip_3yr_avg', 'woba_3yr_avg', 'wrc_plus_3yr_avg', 'hr_3yr_avg', 'rbi_3yr_avg', 'run_3yr_avg', 'hit_3yr_avg', 'double_3yr_avg', 'triple_3yr_avg', 'bb_3yr_avg', 'hp_3yr_avg', 'kk_3yr_avg', 'sb_3yr_avg', 'wpa_3yr_avg', 'war_3yr_avg', 'war_3yr_sum', 'war_last_year', 'ops_last_year', 'wrc_plus_last_year']


In [6]:
X_hitter = hitter_model_df[hitter_features]
y_hitter = hitter_model_df[target]

X_train_hitter, X_test_hitter, y_train_hitter, y_test_hitter = train_test_split(
    X_hitter,
    y_hitter,
    test_size=0.2,
    random_state=42
)

lr_hitter = LinearRegression()
lr_hitter.fit(X_train_hitter, y_train_hitter)

pred_lr = lr_hitter.predict(X_test_hitter)

print("Linear")
print("R2 :", r2_score(y_test_hitter, pred_lr))
print("RMSE :", np.sqrt(mean_squared_error(y_test_hitter, pred_lr)))

rf_hitter = RandomForestRegressor(
    n_estimators=300,
    max_depth=5,
    random_state=42
)

rf_hitter.fit(X_train_hitter, y_train_hitter)

pred_rf = rf_hitter.predict(X_test_hitter)

print("\nRandom Forest")
print("R2 :", r2_score(y_test_hitter, pred_rf))
print("RMSE :", np.sqrt(mean_squared_error(y_test_hitter, pred_rf)))

Linear
R2 : 0.3842446486694411
RMSE : 6.005899511131983

Random Forest
R2 : 0.38779046474833
RMSE : 5.98858211267918


In [7]:
importance_hitter = pd.DataFrame({
    "feature": X_hitter.columns,
    "importance": rf_hitter.feature_importances_
})

importance_hitter = importance_hitter.sort_values(
    by="importance",
    ascending=False
)

display(
    importance_hitter.head(15)
)

,feature,importance
24,war_3yr_sum,0.154708
14,run_3yr_avg,0.121933
16,double_3yr_avg,0.082532
26,ops_last_year,0.062045
10,woba_3yr_avg,0.058386
25,war_last_year,0.050521
1,age_at_fa,0.042930
6,slg_3yr_avg,0.038849
11,wrc_plus_3yr_avg,0.037959
2,games_3yr_avg,0.031655


In [8]:
pitcher_model_df = pitcher.copy()

pitcher_drop_cols = [
    "player_name",
    "position",
    "team",

    "annual_avg_salary",

    "total_contract_amount",
    "contract_years"
]

pitcher_features = [
    col
    for col in pitcher_model_df.columns
    if col not in pitcher_drop_cols
]

for col in pitcher_features + [target]:
    pitcher_model_df[col] = pd.to_numeric(
        pitcher_model_df[col],
        errors="coerce"
    )

for col in pitcher_model_df.columns:
    if pitcher_model_df[col].isnull().sum() > 0:
        pitcher_model_df[col] = pitcher_model_df[col].fillna(
            pitcher_model_df[col].median()
        )

print(pitcher_model_df.shape)

(43, 22)


In [9]:
X_pitcher = pitcher_model_df[pitcher_features]
y_pitcher = pitcher_model_df[target]

X_train_pitcher, X_test_pitcher, y_train_pitcher, y_test_pitcher = train_test_split(
    X_pitcher,
    y_pitcher,
    test_size=0.2,
    random_state=42
)

lr_pitcher = LinearRegression()
lr_pitcher.fit(X_train_pitcher, y_train_pitcher)

pred_lr = lr_pitcher.predict(X_test_pitcher)

print("Linear")
print("R2 :", r2_score(y_test_pitcher, pred_lr))
print("RMSE :", np.sqrt(mean_squared_error(y_test_pitcher, pred_lr)))

rf_pitcher = RandomForestRegressor(
    n_estimators=300,
    max_depth=5,
    random_state=42
)

rf_pitcher.fit(X_train_pitcher, y_train_pitcher)

pred_rf = rf_pitcher.predict(X_test_pitcher)

print("\nRandom Forest")
print("R2 :", r2_score(y_test_pitcher, pred_rf))
print("RMSE :", np.sqrt(mean_squared_error(y_test_pitcher, pred_rf)))

Linear
R2 : 0.7555083561117172
RMSE : 3.295948553689368

Random Forest
R2 : 0.5698608008605024
RMSE : 4.3717259039251335


In [10]:
importance_pitcher = pd.DataFrame({
    "feature": X_pitcher.columns,
    "importance": rf_pitcher.feature_importances_
})

importance_pitcher = importance_pitcher.sort_values(
    by="importance",
    ascending=False
)

display(
    importance_pitcher.head(15)
)

,feature,importance
0,fa_year,0.155314
3,innings_3yr_avg,0.149466
9,hold_3yr_avg,0.129485
4,era_3yr_avg,0.082686
12,war_3yr_sum,0.080853
11,war_3yr_avg,0.062163
10,hit_allowed_3yr_avg,0.050486
6,win_3yr_avg,0.049846
1,age_at_fa,0.043767
7,lose_3yr_avg,0.042284
